# Auslan sign-chat backend on Colab

Starts the chat API (`sign_chat_backend/`) on a Colab GPU and gives it a public URL for the frontend.

- **sign → text**: Uni-Sign, Arm A from `openasl_pose_only_slt.pth` fine-tuned on Auslan-Daily (Communication BLEU-4 18.03)
- **text → sign**: SignSparK fine-tuned on Auslan-Daily Communication, round 2 (hands from retrieved keyframes, σ=1 smoothing)
- **dialogue**: Qwen2.5-1.5B-Instruct by default (switch in section 3)

A T4 is enough; an L4/A100 is faster. First start copies ~14 GB from Drive and downloads mT5 + M-CLIP + Qwen, about 10–15 minutes. The URL lives as long as this runtime.

## 1. Drive, GPU, code

In [1]:
import os, sys, glob, json, shutil, subprocess, time
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU - switch the runtime to a GPU')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
LOCAL = '/content/signchat_local'
os.makedirs(LOCAL, exist_ok=True)

# The backend code: this repository's branch. For a private repo add a Colab secret GITHUB_TOKEN (key icon, left bar).
REPO_URL = 'https://github.com/randlyoyo/FIT5120-TE38-SignLanguage.git'
BRANCH = 'recognition'
APP = '/content/app'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
if not os.path.isdir(f'{APP}/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, url, APP], check=True)
else:
    subprocess.run(['git', '-C', APP, 'pull', '-q'], check=True)
print('backend at', subprocess.run(['git', '-C', APP, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)

Mounted at /content/drive
backend at 6914640 Add the sign-chat backend: an avatar that converses in Auslan



In [2]:
# The two research repos, pinned to the commits the models were trained with
UNISIGN, UNISIGN_COMMIT = '/content/Uni-Sign', 'eed438bcb49e30405cd6ccdfcccca330c134e830'
SSK, SSK_COMMIT = '/content/SignSparK', '22a0b4ec292233c117be09a273fd8577bbbf7d8c'
for path, repo, commit in [(UNISIGN, 'https://github.com/ZechengLi19/Uni-Sign.git', UNISIGN_COMMIT),
                           (SSK, 'https://github.com/JianHe0628/SignSparK.git', SSK_COMMIT)]:
    if not os.path.isdir(f'{path}/.git'):
        subprocess.run(['git', 'clone', '-q', repo, path], check=True)
    subprocess.run(['git', '-C', path, 'checkout', '-q', commit], check=True)

!pip -q install -r /content/app/research/signtest/sign_chat_backend/requirements.txt
!pip -q install wandb==0.21.3 sacrebleu
!apt-get -qq install -y ffmpeg > /dev/null

from huggingface_hub import snapshot_download
MT5 = f'{UNISIGN}/pretrained_weight/mt5-base'
snapshot_download('google/mt5-base', revision='2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f', local_dir=MT5,
                  allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
print('repos and packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 146.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.0/336.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

repos and packages ready


## 2. Weights and data from Drive

Copied to local disk once per runtime (Drive reads during generation are slow).

In [ ]:
UNISIGN_RUN = 'arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16'
SRC = {
    'unisign.pt': f'{WORK}/runs/{UNISIGN_RUN}/checkpoint.pt',
    'signspark_ft_smooth': f'{WORK}/signspark_ft_smooth/final',
    'bank/AuslanDaily_train.lmdb': f'{WORK}/smplx_full/lmdb_smooth/train/AuslanDaily_train.lmdb',
    'SMPLX_NEUTRAL_2020.npz': (glob.glob(f'{DRIVE}/smplx_models/**/SMPLX_NEUTRAL_2020.npz', recursive=True) or [None])[0],
}
missing = [k for k, v in SRC.items() if not v or not os.path.exists(v)]
if missing:
    raise SystemExit(f'missing on Drive: {missing}')
t0 = time.time()
for dst, src in SRC.items():
    out = f'{LOCAL}/{dst}'
    if os.path.exists(out):
        continue
    os.makedirs(os.path.dirname(out), exist_ok=True)
    (shutil.copytree if os.path.isdir(src) else shutil.copyfile)(src, out)
    print('copied', dst)
for s in ('hand', 'body', 'face'):
    assert os.path.exists(f'{LOCAL}/signspark_ft_smooth/{s}.pt'), s
print(f'weights ready ({time.time() - t0:.0f}s)')

copied unisign.pt


## 3. Config and public URL

`DIALOGUE`: `hf` (local Qwen, free), `anthropic` (Claude API; add a Colab secret `ANTHROPIC_API_KEY`), `openai` (any OpenAI-compatible server), or `echo` (the avatar just signs back what it was told - a pure translation mode).

The public URL comes from a Cloudflare quick tunnel: no account, but anyone with the URL can use the API while the runtime is up.

In [ ]:
DIALOGUE = 'hf'
DIALOGUE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct' if DIALOGUE == 'hf' else 'claude-opus-5'
PORT = 8000

if DIALOGUE == 'anthropic':
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# tunnel first, so the server knows its public URL
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
import re
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        PUBLIC_URL = m.group(0)
        break
print('public URL:', PUBLIC_URL)

import yaml
CFG = {
    'device': 'cuda', 'mock': False, 'media_dir': '/content/signchat_media', 'public_base_url': PUBLIC_URL or '',
    'sign2text': {'unisign_repo': UNISIGN, 'mt5_path': MT5, 'checkpoint': f'{LOCAL}/unisign.pt',
                  'code_dir': '/content/app/research/signtest/unisign', 'pose_device': 'cuda'},
    'text2sign': {'signspark_repo': SSK, 'code_dir': '/content/app/research/signtest/auslan_smplx',
                  'weights_dir': f'{LOCAL}/signspark_ft_smooth', 'bank_lmdb': f'{LOCAL}/bank/AuslanDaily_train.lmdb',
                  'smplx_npz': f'{LOCAL}/SMPLX_NEUTRAL_2020.npz'},
    'dialogue': {'backend': DIALOGUE, 'model': DIALOGUE_MODEL},
}
with open('/content/signchat.yaml', 'w') as fh:
    yaml.safe_dump(CFG, fh, sort_keys=False)
print(open('/content/signchat.yaml').read())

## 4. Start the server

In [ ]:
import requests
env = dict(os.environ, SIGNCHAT_CONFIG='/content/signchat.yaml', WANDB_MODE='disabled',
           TOKENIZERS_PARALLELISM='false', TQDM_DISABLE='1')
server_log = open('/content/server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'signchat.server:app', '--host', '0.0.0.0', '--port', str(PORT)],
                          cwd='/content/app/research/signtest/sign_chat_backend', env=env, stdout=server_log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print(open('/content/server.log').read()[-5000:])
        raise SystemExit('server exited')
    try:
        health = requests.get(f'http://localhost:{PORT}/api/health', timeout=2).json()
        break
    except Exception:
        print(f'loading models ... {time.time() - t0:.0f}s', end='\r')
print(json.dumps(health, indent=2))
print('\nAPI:', PUBLIC_URL, '| docs:', f'{PUBLIC_URL}/docs')

## 5. Try it

In [ ]:
r = requests.post(f'http://localhost:{PORT}/api/chat/text', json={'text': 'Hello, how are you?'}).json()
print(json.dumps({k: v for k, v in r.items()}, indent=2)[:2000])
from IPython.display import Video, display
video = r['reply']['sign'].get('video_url')
if video:
    display(Video('/content/signchat_media/' + video.rsplit('/', 1)[1], embed=True, width=480))

In [ ]:
# sign -> text -> reply: upload a signing video (mp4/webm) from your computer
from google.colab import files
up = files.upload()
name = next(iter(up))
with open(name, 'rb') as fh:
    r = requests.post(f'http://localhost:{PORT}/api/chat/sign', files={'video': (name, fh)},
                      data={'mirrored': 'false', 'session_id': 'colab-test'}).json()
print(json.dumps(r, indent=2)[:2000])

In [ ]:
# server log (errors show up here)
print(open('/content/server.log').read()[-4000:])